# Main Figure 4

Frozen drawing logic and saved derived inputs. Outputs: `outputs/main/figure_04`. See reproduction status before release.


In [ ]:
from pathlib import Path
import os, sys
sys.dont_write_bytecode = True
_start = Path.cwd().resolve()
REPO = next((p for q in [_start, *_start.parents] for p in [q, q / "github"]
             if (p / "plot_utils" / "release_runtime.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Run from the repository root or a notebook subdirectory.")
sys.path.insert(0, str(REPO / "plot_utils"))
from release_runtime import configure
configure(REPO)


In [ ]:

from pathlib import Path
import os, sys, json, hashlib, tempfile
sys.dont_write_bytecode=True
PROJECT_ROOT=ROOT=REPO/'figure_source_data/shared'
OUTPUT_ROOT=REPO/'outputs/main/figure_04'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
RUNTIME=OUTPUT_ROOT/'runtime'; RUNTIME.mkdir(exist_ok=True)
FIG=ROOT/'fig'
for key in ['MPLCONFIGDIR','XDG_CACHE_HOME','TEMP','TMP','TMPDIR','NUMBA_CACHE_DIR']:
    os.environ[key]=str(RUNTIME)
tempfile.tempdir=str(RUNTIME)

ERA5_ROOT=ROOT/'ERA5_try'
GRUN_ROOT=ROOT/'GRUN_try'
GLADS_ROOT=ROOT/'GLADS_try'
FIG=ROOT/'fig'


In [ ]:

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, LogNorm, Normalize
from matplotlib.collections import LineCollection
from matplotlib.ticker import ScalarFormatter
from scipy.ndimage import gaussian_filter1d
import cartopy
import cartopy.crs as ccrs
from cartopy.io import shapereader
cartopy.config['cache_dir'] = RUNTIME / 'cartopy'
import geopandas as gpd





plt.rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Liberation Sans']
plt.rcParams['svg.fonttype']='none'


In [ ]:

CACHE=ROOT/'new_fig_try/fig4/calculated_results/product_replacement'
hashes={}

P=GRUN_ROOT/'part5_competing_controls_FORMAL_v1'
V=['forcing_mean_intensity','forcing_mass_24','forcing_drought_months_24','n_candidates','met_onset_span_months','is_many_to_one','sm_pre_3_1','sm_recovery_fixed']
LABEL=['Mean deficit intensity','Cumulative deficit','Drought months','Parent number','Onset-history span','Many-to-one indicator','Pre-onset soil state','Soil-state recovery']
CAT=['Meteorological']*3+['Structure']*3+['Land state']*2
MAP=dict(zip(V,LABEL));GROUP=dict(zip(V,CAT));CORE=[V[i] for i in [0,1,3,4,6,7]]
TREND=dict(zip(CORE,['forcing_intensity','forcing_mass','candidate_density','antecedent_extent','preH_soil_state','soil_recovery_fixed']))


CACHE=ROOT/'new_fig_try/fig4/calculated_results/product_replacement'
A=pd.read_csv(CACHE/'GRUN_A.csv')
B=pd.read_csv(CACHE/'GRUN_B.csv')
C=pd.read_csv(CACHE/'GRUN_C.csv')
D=pd.read_csv(CACHE/'GRUN_D.csv')
E=pd.read_csv(CACHE/'GRUN_E.csv')
qc=json.loads((CACHE/'GRUN_qc.json').read_text());selected=qc['selected']


In [ ]:

PRODUCTS=['GRUN','ERA5','GLDAS']
PC={'GRUN':'#007C78','ERA5':'#CB7935','GLDAS':'#667BA5'}
FOLDERS={'GRUN':GRUN_ROOT,'ERA5':ERA5_ROOT,'GLDAS':GLADS_ROOT}
MODELS={};COEFS={};HEATS={};PAIRS={};CORRS=[]
for product in PRODUCTS:
    base=FOLDERS[product]/'part5_competing_controls_FORMAL_v1'
    MODELS[product]=pd.read_csv(base/'05_FORMAL_HIERARCHICAL_MODEL.csv').set_index('model').loc[['F','F+S','F+S+B']]
    COEFS[product]=pd.read_csv(base/'05_FORMAL_STANDARDIZED_COEFFICIENTS.csv').merge(pd.read_csv(base/'05_FORMAL_BLOCK_BOOTSTRAP_SUMMARY.csv'),on='driver',validate='one_to_one')
    HEATS[product]=pd.read_csv(CACHE/(product+'_B_extension.csv'))
    PAIRS[product]=pd.read_csv(CACHE/(product+'_D_extension.csv'))


In [ ]:

BLUE='#337CA3';RED='#C06446';DARK='#303840';MID='#7E898E';DIV=LinearSegmentedColormap.from_list('assoc',[BLUE,'#F5F4F0',RED]);DIV.set_bad('#E5E8EA')
plt.rcParams.update({'font.family':'Arial','font.size':8.2,'axes.labelsize':8.2,'xtick.labelsize':7.4,'ytick.labelsize':7.4,'axes.linewidth':.65,'axes.spines.top':False,'axes.spines.right':False,'lines.linewidth':1.15,'pdf.fonttype':42,'svg.fonttype':'none','text.color':DARK,'axes.labelcolor':DARK})
sel=qc['selected'];short={'forcing_mean_intensity':'Deficit intensity','forcing_mass_24':'Cumulative deficit','forcing_drought_months_24':'Drought months','n_candidates':'Parent number','met_onset_span_months':'History span','is_many_to_one':'Many-to-one','sm_pre_3_1':'Pre-onset soil','sm_recovery_fixed':'Soil recovery'}
core=['forcing_mean_intensity','forcing_mass_24','n_candidates','met_onset_span_months','sm_pre_3_1','sm_recovery_fixed'];tmap=dict(zip(core,['forcing_intensity','forcing_mass','candidate_density','antecedent_extent','preH_soil_state','soil_recovery_fixed']));colour=lambda v:BLUE if v<0 else RED if v>0 else MID
fig=plt.figure(figsize=(7.20472440945,224/25.4),facecolor='white');gs=fig.add_gridspec(2,2,left=.047,right=.986,bottom=.115,top=.925,width_ratios=[1.02,.98],height_ratios=[1,1.06],wspace=.23,hspace=.32)
def title(spec,label,text):
 b=spec.get_position(fig);fig.text(b.x0-.022,b.y1+.016,label,fontsize=12,fontweight='bold',va='bottom');fig.text(b.x0,b.y1+.016,text,fontsize=9.6,fontweight='bold',va='bottom')



fig.legend(handles=[Line2D([],[],color=PC[p],marker='o',label=p,lw=1) for p in PRODUCTS],loc='upper center',bbox_to_anchor=(.5,.99),ncol=3,frameon=False,fontsize=8)


In [ ]:

ag=gs[0,0].subgridspec(1,2,width_ratios=[1.23,1],wspace=.12);a1=ag[0].subgridspec(1,2,width_ratios=[1.05,1],wspace=.015);lab=fig.add_subplot(a1[0]);ax=fig.add_subplot(a1[1]);lab.axis('off');y=.9
for cat in ['Meteorological','Structure','Land state']:
 lab.text(.98,y,cat,fontsize=7.3,fontweight='bold',ha='right',va='center');y+=.72
 for _,r in A[A.category==cat].sort_values('rho',key=abs,ascending=False).iterrows():
  lab.text(.98,y,short[r.variable],fontsize=7.2,ha='right',va='center');ax.plot([r.ci_low,r.ci_high],[y,y],color=colour(r.rho),lw=1.25);ax.plot(r.rho,y,'o',color=colour(r.rho),ms=3.8);y+=.86
 y+=.42
lab.set(xlim=(0,1),ylim=(y-.25,-.4));ax.set(xlim=(-.015,.17),ylim=(y-.25,-.4),xlabel='Partial Spearman ρ');ax.xaxis.label.set_size(7.4);ax.set_yticks([]);ax.set_xticks([0,.08,.16]);ax.axvline(0,color='#B6BEC1',ls='--',lw=.6);ax.spines['left'].set_visible(False);lab.text(0,-.03,'a1  Event scale · GRUN',fontsize=8,fontweight='bold')



ax.set_xlim(min(-.015,float(A.ci_low.min())-.01),max(.17,float(A.ci_high.max())+.015))


In [ ]:

ar=ag[1].subgridspec(2,1,height_ratios=[.8,1.2],hspace=.32)
ax=fig.add_subplot(ar[0])
for j,product in enumerate(PRODUCTS):
 vals=MODELS[product].spatial_block_cv_r2.to_numpy()
 ax.bar(np.arange(3)+(j-1)*.22,vals,color=PC[product],width=.20)
ax.axhline(0,color='.6',lw=.5)
ax.set_ylabel('Spatial CV R²')
ax.set_xticks(range(3),['Met.','+ Structure','+ Land'],fontsize=6.7)
ax.text(0,1.055,'a2  Nested grid-trend models',transform=ax.transAxes,fontsize=8,fontweight='bold')
a3=ar[1].subgridspec(1,2,width_ratios=[1.02,1],wspace=.015)
lab=fig.add_subplot(a3[0]);ax=fig.add_subplot(a3[1]);lab.axis('off')
for i,key in enumerate(core):
 lab.text(.98,i,short[key],ha='right',va='center',fontsize=7)
 for j,product in enumerate(PRODUCTS):
  row=COEFS[product].set_index('driver').loc[tmap[key]+'_sen_decade']
  y=i+(j-1)*.19
  ax.plot([row.ci_low,row.ci_high],[y,y],color=PC[product],lw=.9)
  ax.plot(row.standardized_beta,y,'o',color=PC[product],ms=2.8)
lab.set(xlim=(0,1),ylim=(5.65,-.75));ax.set_ylim(5.65,-.75)
ax.set_xlabel('Standardized β',fontsize=7.4);ax.set_yticks([])
ax.axvline(0,color='#B6BEC1',ls='--',lw=.6);ax.spines['left'].set_visible(False)
lab.text(0,1.055,'a3  Joint coefficients',transform=lab.transAxes,fontsize=8,fontweight='bold')
title(gs[0,0],'a','Event-scale and joint statistical associations')


In [ ]:

bg=gs[0,1].subgridspec(2,4,width_ratios=[1.22,1,1,1],height_ratios=[1,.040],wspace=.06,hspace=.17)
lab=fig.add_subplot(bg[0,0]);lab.axis('off')
for i,key in enumerate(core):lab.text(.98,i,short[key],ha='right',va='center',fontsize=7)
lab.set(xlim=(0,1),ylim=(5.5,-.5))
for j,product in enumerate(PRODUCTS):
 ax=fig.add_subplot(bg[0,j+1])
 heat=HEATS[product].pivot(index='variable',columns='quantile',values='rho').loc[core]
 im=ax.imshow(heat,aspect='auto',cmap=DIV,vmin=-1,vmax=1)
 ax.set_title(product,color=PC[product],fontsize=8,pad=4)
 ax.set_yticks([]);ax.set_xticks(range(5),['10','25','50','75','90'],fontsize=6)
 ax.set_xlabel('PT quantile (%)',fontsize=6.5,labelpad=2);ax.tick_params(length=0)
 ax.axhline(1.5,color='white',lw=3);ax.axhline(3.5,color='white',lw=3)
 for sp in ax.spines.values():sp.set_visible(False)
cax=fig.add_subplot(bg[1,1:])
cb=fig.colorbar(im,cax=cax,orientation='horizontal',ticks=[-1,0,1])
cb.set_label('AR6 trend association (Spearman ρ)',labelpad=3)
title(gs[0,1],'b','Quantile-specific associations')


In [ ]:

cmid=gs[1,0].subgridspec(1,3,width_ratios=[.07,.86,.07],wspace=0)[1];cg=cmid.subgridspec(3,1,hspace=.40);rcols={'Short':RED,'Median':MID,'Long':BLUE};axes=[]
for i,key in enumerate(sel):
 ax=fig.add_subplot(cg[i]);axes.append(ax)
 for regime,c in rcols.items():z=C[(C.variable==key)&(C.regime==regime)];ax.plot(z.year,z.rolling10,color=c,lw=1.35 if regime!='Median' else .9,label=regime)
 ax.axhline(0,color='#C7CED1',ls='--',lw=.55);ax.set_xlim(1952,2014);ax.margins(y=.28);ax.text(.015,.93,short[key],transform=ax.transAxes,va='top',fontsize=7.5);ax.set_xticks([1960,1980,2000]);ax.tick_params(labelbottom=i==2)
 if i==1:ax.set_ylabel('Standardized anomaly')
 if i==2:ax.set_xlabel('Year')
axes[0].legend(frameon=False,ncol=3,loc='lower center',fontsize=7,handlelength=1.4,columnspacing=.8);title(gs[1,0],'c','Temporal co-variation · GRUN')




In [ ]:
fit_lines=pd.read_csv(REPO/'figure_source_data/main/figure_04/regional_fit_lines.csv',float_precision='round_trip')
fit_correlations=pd.read_csv(CACHE/'three_product_regional_correlations.csv')

dmid=gs[1,1].subgridspec(1,3,width_ratios=[.10,.80,.10],wspace=0)[1]
dg=dmid.subgridspec(3,1,hspace=.48)
all_y=pd.concat([PAIRS[p] for p in PRODUCTS]).propagation_slope
for i,key in enumerate(sel):
 ax=fig.add_subplot(dg[i]);rho_text=[]
 for product in PRODUCTS:
  z=PAIRS[product][PAIRS[product].variable==key].dropna()
  x=z.candidate_slope.to_numpy();y=z.propagation_slope.to_numpy()
  saved=fit_correlations[(fit_correlations['product']==product)&(fit_correlations.variable==key)].iloc[0];rho,pval=saved.rho,saved.p
  line=fit_lines[(fit_lines['product']==product)&(fit_lines.variable==key)].sort_values('point_index');xx=line.x.to_numpy();yy=line.y.to_numpy();assert len(line)==100
  ax.scatter(x,y,s=13 if product=='GRUN' else 10,color=PC[product],alpha=.6,edgecolors='white',linewidths=.25)
  ax.plot(xx,yy,color=PC[product],lw=1 if product=='GRUN' else .8)
  rho_text.append(f'{product} {rho:+.2f}')
  CORRS.append(dict(product=product,variable=key,rho=rho,p=pval,n_regions=len(z)))
 ax.axhline(0,color='#B9C1C5',lw=.5,ls='--');ax.axvline(0,color='#D1D5D7',lw=.45)
 ax.text(.015,.96,short[key]+'  ρ: '+', '.join(rho_text),transform=ax.transAxes,va='top',fontsize=6.1)
 ax.set_ylim(all_y.min()-.05,all_y.max()+.09)
 ax.set_ylabel('TP slope')
 ax.set_xlabel({'forcing_mass_24':'Cumulative deficit trend','met_onset_span_months':'History-span trend','sm_recovery_fixed':'Soil-recovery trend'}[key],labelpad=1.5)
title(gs[1,1],'d','Regional trend correspondence')
fig.text(.047,.022,'1952–2014. a1/c: GRUN event cohort; a2/a3/b/d: GRUN, ERA5, GLDAS with identical statistical definitions.\na1: AR6/year-adjusted rank associations and 95% spatial-block intervals. a2/a3: existing formal spatial-CV and coefficient results.\nb/d: AR6 trend correspondence, not causal attribution. c: original annual percentile bands, equal-grid means and centred 10-year curves.\nLand-state covariates retain the existing soil input; runoff-product replacement does not create a new soil-moisture product.',fontsize=7,linespacing=1.35)


In [ ]:

FIGURE_NUMBER=4
FROZEN_TITLES={'a': 'Associations accompany limited spatial predictive power', 'b': 'Associations differ across propagation-time quantiles', 'c': 'GRUN timing regimes show differing covariation', 'd': 'Regional trend correspondence varies across products'}
from matplotlib.text import Text
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import textwrap
OUTDIR=Path(OUT if FIGURE_NUMBER==6 else OUTPUT_ROOT)
QA=OUTDIR/'qa';QA.mkdir(exist_ok=True)
def numerical_artists():
    result={}
    def add(key,value):
        a=np.ma.asarray(value)
        if a.dtype.kind in 'biufc':
            v=np.ascontiguousarray(a.filled(np.nan) if np.ma.isMaskedArray(a) and a.dtype.kind in 'fc' else a)
            result[key]={'shape':list(v.shape),'sha256':hashlib.sha256(v.tobytes()).hexdigest()}
    for j,a in enumerate(fig.axes):
        for k,l in enumerate(a.lines):add(f'{j}/line{k}/x',l.get_xdata());add(f'{j}/line{k}/y',l.get_ydata())
        for k,l in enumerate(a.collections):
            if l.get_array() is not None:add(f'{j}/collection{k}/array',l.get_array())
            if hasattr(l,'get_offsets'):add(f'{j}/collection{k}/offsets',l.get_offsets())
            if hasattr(l,'get_segments'):
                for h,v in enumerate(l.get_segments()):add(f'{j}/collection{k}/segment{h}',v)
            if l.__class__.__name__ in ('PolyCollection','FillBetweenPolyCollection'):
                for h,v in enumerate(l.get_paths()):add(f'{j}/collection{k}/path{h}',v.vertices)
        for k,l in enumerate(a.images):add(f'{j}/image{k}',l.get_array())
        for k,l in enumerate(a.patches):
            if isinstance(l,matplotlib.patches.Rectangle):add(f'{j}/bar{k}',[l.get_x(),l.get_y(),l.get_width(),l.get_height()])
    return result
fig.canvas.draw()
before=numerical_artists()
contracts=json.loads((REPO/'assets/other/panel_contracts.json').read_text(encoding='utf-8'))
oldtitles={r[2] for r in contracts if r[0]==FIGURE_NUMBER}
notes=[]
for t in list(fig.findobj(Text)):
    s=t.get_text()
    if s in oldtitles or s in list('abcdef'):
        t.set_text('');continue
    if len(s)>180:
        notes.append(s);t.set_text('');continue
    s=s.replace('decade$^{-1}$','decade⁻¹').replace(r'$\beta$','β').replace('$R^2$','R²')
    t.set_text(s);t.set_fontfamily('DejaVu Sans');t.set_fontsize(5.5 if FIGURE_NUMBER in (1,2) else 6)
    if t.get_rotation():t.set_rotation_mode('anchor')
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial','Liberation Sans'],
    'font.size':6,'axes.labelsize':6,'xtick.labelsize':5.5,'ytick.labelsize':5.5,
    'legend.fontsize':5.5,'pdf.fonttype':42,'svg.fonttype':'none'})
DIV=plt.get_cmap('RdBu_r')
for ax in fig.axes:
    cb=getattr(ax,'_colorbar',None)
    for sp in ax.spines.values():sp.set_linewidth(.45);sp.set_edgecolor('#657076')
    ax.tick_params(width=.45,length=2,pad=2,labelsize=5.5)
    ax.xaxis.label.set_fontsize(6);ax.yaxis.label.set_fontsize(6)
    ax.title.set_fontsize(6)
    if cb is not None:
        cb.outline.set_linewidth(.35)
        ax.tick_params(length=1.5,pad=1)
        ax.set_axes_locator(None)
        p=ax.get_position()
        if cb.orientation=='horizontal':ax.set_position([p.x0,p.y0,p.width,min(p.height,2.5/(72*fig.get_figheight()))])
        else:ax.set_position([p.x0,p.y0,min(p.width,2.5/(72*fig.get_figwidth())),p.height])
    for artist in list(ax.images)+list(ax.collections):
        norm=getattr(artist,'norm',None)
        if norm is not None and norm.vmin is not None and norm.vmax is not None and norm.vmin<0<norm.vmax:
            assert np.isclose(abs(norm.vmin),abs(norm.vmax)), 'Original diverging limits require review'
            artist.set_cmap(DIV);artist.set_norm(TwoSlopeNorm(vmin=norm.vmin,vcenter=0,vmax=norm.vmax))
    for line in ax.lines:
        if line.get_linestyle() in ('--',':'):
            x=np.asarray(line.get_xdata());y=np.asarray(line.get_ydata())
            if (len(x)>0 and np.all(x==0)) or (len(y)>0 and np.all(y==0)):
                line.set_color('#8B9298');line.set_linewidth(.5);line.set_linestyle('--')
        if line.get_markersize()>4:line.set_markersize(3.5)
        if line.get_linewidth()>1.8:line.set_linewidth(1.5)
    if hasattr(ax,'projection'):
        for col in ax.collections:
            if col.__class__.__name__=='FeatureArtist':col.set_linewidth(.25)
        ax.spines['geo'].set_linewidth(.45)

if FIGURE_NUMBER==1:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='24 mo':t.set_x(.62);t.set_transform(ax.transAxes);t.set_y(.07);t.set_ha('right')
            if t.get_text().startswith('d1  '):t.set_position((0,1.02));t.set_va('bottom')
            if t.get_text().startswith('Common 36-month'):
                notes.append(t.get_text());t.set_text('Common 36-month\ncoverage cohort');t.set_position((.04,.06));t.set_fontsize(5)
if FIGURE_NUMBER==2:
    for ax in fig.axes:
        if ax.get_xlabel()=='Months relative to meteorological anchor':ax.set_xlabel('Months from meteorological anchor',fontsize=6)
        for t in ax.texts:
            if t.get_text().startswith('Negative ='):notes.append(t.get_text());t.set_text('')
            if '95% block CI' in t.get_text():t.set_position((.03,1.02));t.set_va('bottom');t.set_fontsize(5)
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb is not None and 'hexagon' in ax.get_xlabel():
            from matplotlib.ticker import FixedLocator,FixedFormatter,NullLocator
            ax.xaxis.set_major_locator(FixedLocator([1,10,100,1000]));ax.xaxis.set_major_formatter(FixedFormatter(['1','10','100','1000']));ax.xaxis.set_minor_locator(NullLocator())
if FIGURE_NUMBER==3:
    for leg in list(fig.legends):
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            handles=leg.legend_handles;labels=[t.get_text() for t in leg.get_texts()]
            pos=gs[0,0].get_position(fig);leg.remove()
            fig.legend(handles,labels,ncol=4,frameon=False,fontsize=5.2,loc='upper left',bbox_to_anchor=(pos.x0-.005,pos.y0-.038),handlelength=1.2,columnspacing=.8)
    maps=[a for a in fig.axes if hasattr(a,'projection')]
    for a in maps:
        a.title.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Main widths pool events'):
            t.set_text('Main widths: pooled events. d2/f: grid-year means. Grey: no estimate.');t.set_fontsize(5.5)
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Thin: annual'):
                t.set_position((.02,1.01));t.set_va('bottom');t.set_text('Thin: annual; thick: 5-year visual mean');t.set_fontsize(5)
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='horizontal':
            p=ax.get_position();ax.set_position([p.x0,p.y0-.01,p.width,p.height]);cb.set_ticks([cb.norm.vmin,0,cb.norm.vmax] if cb.norm.vmin<0 else [0,50,100])
        if ax.images:
            p=ax.get_position();ax.set_position([p.x0+.04,p.y0,p.width-.04,p.height]);ax.tick_params(axis='y',labelsize=5)
if FIGURE_NUMBER==4:
    for leg in fig.legends:leg.set_bbox_to_anchor((.5,.045))
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text().replace('Parent number','Antecedent count').replace('Many-to-one','Multiple antecedents')
            if '  ρ: ' in s:
                s=s.replace('  ρ: ','\nρ: ');t.set_position((.01,1.03));t.set_va('bottom');t.set_fontsize(5.2)
            if s.startswith('a1  '):
                t.set_text('');p=ax.get_position();fig.text(p.x0,p.y1+.013,'a1  Event scale · GRUN',fontsize=5.5,fontweight='bold')
            else:t.set_text(s)
            if not ax.axison:t.set_fontsize(5.2)
        if ax.get_ylabel()=='TP slope':ax.set_ylabel('PT slope',fontsize=6)
        cb=getattr(ax,'_colorbar',None)
        if cb is not None:cb.set_ticks([-1,0,1])
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text()
            if s.startswith('Envelope spans') or s.startswith('Filled:'):
                t.set_position((0,-.22));t.set_ha('left');t.set_va('top');t.set_fontsize(5.2)
            if s=='Severity decile':t.set_x(-.29);t.set_fontsize(5.5)
            if 'median β < 0' in s:
                t.set_position((.0,-.32));t.set_ha('left');t.set_va('top');t.set_text(s.replace('\n','; ',1));t.set_fontsize(5.2);t.set_bbox(None)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._loc=2
            for cax in fig.axes:
                cb=getattr(cax,'_colorbar',None)
                if cb and cb.orientation=='horizontal':
                    p=ax.get_position();cax.set_position([p.x0,p.y0-.105,p.width,2.5/(72*fig.get_figheight())])
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='threshold floor':t.set_text('AUC < 0.60: L* = 0');t.set_x(.34);t.set_fontsize(5.5)
            if t.get_text().startswith('Δ ='):
                t.set_x(1.0);t.set_fontsize(5);t.set_bbox(None)
                if ax.get_title()=='Retrospective PT':
                    t.set_x(.02);t.set_ha('left');t.set_text(t.get_text().replace('Δ = ','Δ '))
            if t.get_text() in ('GRUN','ERA5','GLDAS') and ax in fig.axes[12:]:
                t.set_position((.50,.84));t.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Points: temporal'):t.set_y(.025);t.set_fontsize(5)

if FIGURE_NUMBER in (1,2,3):specs=[gs[0,0],gs[0,1],gs[0,2],gs[1,0],gs[1,1],gs[1,2]]
elif FIGURE_NUMBER==4:specs=[gs[0,0],gs[0,1],gs[1,0],gs[1,1]]
elif FIGURE_NUMBER==5:specs=[outer[0,0],outer[0,1],outer[0,2],outer[1,0],outer[1,1],outer[1,2]]
else:specs=top+bottom
headers=[]
for letter,spec in zip('abcdef',specs):
    b=spec.get_position(fig);h=fig.get_figheight()*72;w=fig.get_figwidth()*72
    width_chars=33 if FIGURE_NUMBER in (1,2) else 48
    title_offset=9 if FIGURE_NUMBER in (1,2) else 24 if FIGURE_NUMBER==4 else 18
    txt=textwrap.fill(FROZEN_TITLES[letter],width_chars,break_long_words=False)
    t=fig.text(b.x0,b.y1+title_offset/h,txt,ha='left',va='bottom',fontsize=6.5,fontweight='normal',linespacing=1.4,color='#20272B')
    fig.text(b.x0-10/w,b.y1+title_offset/h,letter,ha='left',va='bottom',fontsize=8,fontweight='bold',color='#20272B')
    headers.append(t)
fig.canvas.draw()
after=numerical_artists()
assert before==after,'A scientific artist changed during styling'
(QA/'numerical_artist_invariance.json').write_text(json.dumps({'status':'PASS','entries':len(before),'before':before,'after':after},indent=2),encoding='utf-8')
(QA/'retained_caption_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
from audit_panel_alignment import require_matplotlib_panel_alignment
eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None and not hasattr(a,'projection')]
if len(eligible)<2:eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None]
ids=[f'plot{i+1}' for i in range(len(eligible))]
rows={};cols={}
for ident,ax in zip(ids,eligible):
    p=ax.get_position();rows.setdefault((round(p.y0,5),round(p.y1,5)),[]).append(ident);cols.setdefault((round(p.x0,5),round(p.x1,5)),[]).append(ident)
rg=[v for v in rows.values() if len(v)>1];cg=[v for v in cols.values() if len(v)>1]
ex=[]
for group in cg:
    if len(group)>=3:
        parent_ids={str(eligible[ids.index(k)].get_subplotspec().get_gridspec()) for k in group}
        ex.append({'panels':group,'checks':['vertical-gutter'],'reason':'Column spans distinct main-panel rows and nested stacks; outer and inner gutters intentionally differ. Stack spacing remains uniform within each original nested GridSpec.'})
for group in rg:
    if len(group)>=3:
        widths=[eligible[ids.index(k)].get_position().width for k in group]
        if np.ptp(widths)>1e-4:
            ex.append({'panels':group,'checks':['panel-width','horizontal-gutter'],'reason':'Nested composites intentionally allocate widths to distinct quantities; within-product repeats are compared separately.'})
require_matplotlib_panel_alignment(fig,axes=eligible,panel_ids=ids,row_groups=rg,column_groups=cg,
    exemptions=ex,require_panel_labels=False,json_out=QA/'alignment.json',overlay_svg=QA/'alignment.svg',strict=True)
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.png',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.pdf',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_Nature_final.svg',dpi=450,facecolor='white')
fig.canvas.draw()
meta=[{'axis':j,'bounds':list(a.get_position().bounds),'title':a.get_title(),'xlabel':a.get_xlabel(),'ylabel':a.get_ylabel(),
       'texts':[t.get_text() for t in a.texts]} for j,a in enumerate(fig.axes)]
(QA/'axes.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'Figure {FIGURE_NUMBER}: exported, {len(before)} numerical artist arrays unchanged.')
plt.close(fig)
